# Dependents index statistics

This notebook computes the statistics of the dependents index, which is built from the Goblin snapshot of Maven Central (20 April 2025).

It shows the counts after each filtering step, the per library statistics for the whole index and for libraries with at least 50 popular dependents, and prints the LaTeX table used in the paper.

A popular dependent is a dependent project that at least one other project depends on (reach > 0).

**Needs** the index in `data/dependents-index` (see the pipeline README, `publish --download`) and `pip install -r requirements.txt`. The queries read about 136 million edges and will take time and resources. 

In [3]:
!pip3 install duckdb==2.0.0

In [4]:
import os
from pathlib import Path

import duckdb

# The index folder. Set OSSBRIDGE_DATA to use another data folder.
DATA = Path(os.environ.get("OSSBRIDGE_DATA", Path.cwd().parent / "data"))
INDEX = DATA / "dependents-index"
MIN_POPULAR = 50

con = duckdb.connect()
con.execute("SET memory_limit = '10GB'")
print(INDEX)

/Users/rachnaraj/Documents/FSE-2026-Tool/OSSBridge/data/dependents-index


## Filtering steps

0. **Raw Goblin graph.** Every dependency edge from a dependent release to a library version.
1. **Remove test-scope dependencies.** Test code is not in the published jar, so its calls cannot be analyzed.
2. **Remove inexact versions.** Ranges, properties, `LATEST`, `RELEASE` and `SNAPSHOT` do not tell which version is used. The pattern is the one in `DependentsIndex/build_index.py`.
3. **Remove duplicate edges.** The same release can declare the same library version twice with different scopes.
4. **Keep libraries with at least 50 popular dependents.**

In [5]:
NOT_A_RELEASE = r"[][(),${}\s]|^(RELEASE|LATEST)$|SNAPSHOT"

con.execute(f"CREATE OR REPLACE VIEW step0 AS SELECT * FROM read_parquet('{INDEX / 'edges.parquet'}')")
con.execute("CREATE OR REPLACE TABLE step1 AS SELECT * FROM step0 WHERE coalesce(scope, '') <> 'test'")
con.execute(f"CREATE OR REPLACE TABLE step2 AS SELECT * FROM step1 WHERE NOT regexp_matches(library_version, '{NOT_A_RELEASE}')")
con.execute("""
    CREATE OR REPLACE TABLE step3 AS
    SELECT DISTINCT library_ga, library_version, dependent_ga, dependent_version FROM step2""")

# Reach of each project is the number of distinct projects that directly depend on it.
con.execute(f"""
    CREATE OR REPLACE TABLE reach AS
    SELECT ga, max(reach) AS reach FROM read_parquet('{INDEX / 'reach.parquet'}') GROUP BY ga""")

# One row per library after step 3, with its versions, dependents and popular dependents.
con.execute("""
    CREATE OR REPLACE TABLE libraries AS
    SELECT s.library_ga,
           count(DISTINCT s.library_version) AS versions,
           count(DISTINCT s.dependent_ga) AS dependents,
           count(DISTINCT s.dependent_ga) FILTER (WHERE r.reach > 0) AS popular
    FROM step3 s LEFT JOIN reach r ON r.ga = s.dependent_ga
    GROUP BY s.library_ga""")
con.execute(f"""
    CREATE OR REPLACE TABLE step4 AS
    SELECT s.* FROM step3 s JOIN libraries l USING (library_ga) WHERE l.popular >= {MIN_POPULAR}""")

In [6]:
def counts(step, label):
    return f"""
        SELECT '{label}' AS step, count(*) AS edges,
               count(DISTINCT library_ga) AS libraries,
               count(DISTINCT (library_ga, library_version)) AS library_versions,
               count(DISTINCT dependent_ga) AS dependent_projects,
               count(DISTINCT (dependent_ga, dependent_version)) AS dependent_releases
        FROM {step}"""

funnel = con.sql(" UNION ALL ".join([
    counts("step0", "0. Raw Goblin graph"),
    counts("step1", "1. Remove test-scope dependencies"),
    counts("step2", "2. Remove inexact versions"),
    counts("step3", "3. Remove duplicate edges"),
    counts("step4", f"4. Keep libraries with >= {MIN_POPULAR} popular dependents"),
]) + " ORDER BY step")
funnel

stepVARCHAR,edgesBIGINT,librariesBIGINT,library_versionsBIGINT,dependent_projectsBIGINT,dependent_releasesBIGINT
0. Raw Goblin graph,135819916,339450,8098948,599087,14408454
1. Remove test-scope dependencies,105554410,326665,7778305,582089,14049634
2. Remove inexact versions,104145156,307987,7634985,563767,13842490
3. Remove duplicate edges,104127133,307987,7634985,563767,13842490
4. Keep libraries with >= 50 popular dependents,67837552,4082,343160,508360,12354144


## Popular dependent projects

In [7]:
con.sql("""
    SELECT 'Whole index' AS scope, count(DISTINCT s.dependent_ga) AS popular_dependent_projects
    FROM step3 s JOIN reach r ON r.ga = s.dependent_ga WHERE r.reach > 0
    UNION ALL
    SELECT 'At least 50 popular dependents', count(DISTINCT s.dependent_ga)
    FROM step4 s JOIN reach r ON r.ga = s.dependent_ga WHERE r.reach > 0""")

scopeVARCHAR,popular_dependent_projectsBIGINT
Whole index,279579
At least 50 popular dependents,253833


## Per library statistics

Versions per library counts the versions that have at least one dependent after step 3.

In [8]:
def per_library(where):
    return con.execute(f"""
        SELECT count(*), sum(versions),
               min(versions), avg(versions), median(versions), max(versions),
               min(popular), avg(popular), median(popular), max(popular),
               min(dependents), avg(dependents), median(dependents), max(dependents)
        FROM libraries {where}""").fetchone()

whole = per_library("")
kept = per_library(f"WHERE popular >= {MIN_POPULAR}")

names = ["libraries", "library versions",
         "versions min", "versions mean", "versions median", "versions max",
         "popular min", "popular mean", "popular median", "popular max",
         "dependents min", "dependents mean", "dependents median", "dependents max"]
for name, w, k in zip(names, whole, kept):
    print(f"{name:20} {w:>14,.2f} {k:>14,.2f}")

libraries                307,987.00       4,082.00
library versions       7,634,985.00     343,160.00
versions min                   1.00           1.00
versions mean                 24.79          84.07
versions median                5.00          35.00
versions max               2,959.00       2,772.00
popular min                    0.00          50.00
popular mean                   6.88         339.46
popular median                 1.00         103.00
popular max               50,807.00      50,807.00
dependents min                 1.00          50.00
dependents mean               12.55         611.84
dependents median              2.00         181.00
dependents max            87,758.00      87,758.00


## Top 10 libraries by popular dependents

In [ ]:
con.sql(f"""
    SELECT library_ga, popular, dependents, versions FROM libraries
    WHERE popular >= {MIN_POPULAR} ORDER BY popular DESC, library_ga LIMIT 10""")

ResourceInUseException: Resource In Use Error: connection has an unfinished result from execute(); call complete() or abort(), or fetch until a fetch returns no rows, before other work on this connection, or use a separate connection from cursor()

ResourceInUseException: Resource In Use Error: connection has an unfinished result from execute(); call complete() or abort(), or fetch until a fetch returns no rows, before other work on this connection, or use a separate connection from cursor()

: 